# 05 — Error Analysis

Goal: compare where the three tiers (TF-IDF+LogReg, BiLSTM, BERT) fail on the **same set of validation examples**, so differences reflect the model, not the sample.

Steps:
1. Load each model's predictions on the SST-2 (and/or SST-5) validation split.
2. Pick one fixed random sample of misclassified examples that overlaps across models where possible.
3. Hand-label each error with a category (negation, intensifier, neutral-adjacent, sarcasm/irony, parse ambiguity, other).
4. Compare error-category distributions across models.
5. Pull out a handful of representative failure cases with commentary.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RANDOM_SEED = 42
DATASET = "sst2"  # or "sst5"

# Each of these should be a DataFrame with columns: ['sentence', 'y_true', 'y_pred']
# on the SAME validation split, one per model. Load however your notebooks 02-04
# saved predictions (e.g. csv/parquet in results/).
#
# baseline_df = pd.read_csv(f"results/baseline_{DATASET}_predictions.csv")
# birnn_df    = pd.read_csv(f"results/birnn_{DATASET}_predictions.csv")
# bert_df     = pd.read_csv(f"results/bert_{DATASET}_predictions.csv")

baseline_df = None  # TODO
birnn_df = None     # TODO
bert_df = None       # TODO

models = {
    "baseline": baseline_df,
    "birnn": birnn_df,
    "bert": bert_df,
}

## 1. Pull misclassified examples per model

In [2]:
def get_errors(df):
    """Return only the rows where the model got it wrong."""
    return df[df["y_true"] != df["y_pred"]].reset_index(drop=True)

error_dfs = {name: get_errors(df) for name, df in models.items() if df is not None}
for name, edf in error_dfs.items():
    print(f"{name}: {len(edf)} errors out of validation set")

## 2. Sample 50 examples per model (same seed → maximizes overlap for shared-sentence comparison)

Sampling with the same `random_state` from the same underlying validation set means indices
line up when multiple models err on the same sentence, which is what lets you distinguish
*persistent* errors (all three models miss it) from *fixed-by-architecture* errors (only the
baseline misses it).

In [3]:
samples = {
    name: edf.sample(min(50, len(edf)), random_state=RANDOM_SEED)
    for name, edf in error_dfs.items()
}

for name, s in samples.items():
    print(f"--- {name} sample ---")
    display(s.head())

## 3. Categorize errors

Hand-label each sampled error. Categories, based on the project's core NLP concepts:

- `negation` — e.g. "not bad", "isn't great"
- `intensifier` — degree modifiers that shift a class boundary ("somewhat", "utterly")
- `neutral_adjacent` — SST-5 boundary cases between neutral and mildly pos/neg
- `sarcasm_irony` — sentiment expressed non-literally
- `parse_ambiguity` — long/compound sentences with mixed clauses
- `other` — anything that doesn't fit cleanly

Add a `category` column by hand (or via a small labeling loop) to each sample DataFrame.

In [4]:
CATEGORIES = [
    "negation", "intensifier", "neutral_adjacent",
    "sarcasm_irony", "parse_ambiguity", "other",
]

def label_errors_interactively(df):
    """
    Simple manual-labeling loop. Run in a notebook cell; type the category
    name (or its index) for each sentence, or 's' to skip/stop early.
    """
    labels = []
    for i, row in df.iterrows():
        print(f"\nSentence: {row['sentence']}")
        print(f"True: {row['y_true']}  |  Pred: {row['y_pred']}")
        for idx, cat in enumerate(CATEGORIES):
            print(f"  [{idx}] {cat}")
        choice = input("Category (index) or 's' to stop: ")
        if choice == "s":
            break
        labels.append(CATEGORIES[int(choice)])
    df = df.iloc[: len(labels)].copy()
    df["category"] = labels
    return df

# Example usage (run per model, this is interactive and manual by design):
# baseline_labeled = label_errors_interactively(samples["baseline"])
# baseline_labeled.to_csv("results/baseline_error_categories.csv", index=False)

## 4. Compare error-category distributions across models

In [5]:
# Once each model's sample has a 'category' column (loaded from the CSVs saved above):
#
# labeled = {
#     "baseline": pd.read_csv("results/baseline_error_categories.csv"),
#     "birnn": pd.read_csv("results/birnn_error_categories.csv"),
#     "bert": pd.read_csv("results/bert_error_categories.csv"),
# }
#
# counts = pd.DataFrame({
#     name: df["category"].value_counts().reindex(CATEGORIES, fill_value=0)
#     for name, df in labeled.items()
# })
#
# counts.plot(kind="bar", figsize=(9, 5))
# plt.ylabel("Number of errors (out of 50 sampled)")
# plt.title(f"Error category distribution by model — {DATASET}")
# plt.tight_layout()
# plt.savefig(f"results/error_categories_{DATASET}.png")
# plt.show()

## 5. Representative failure cases

Pull 3-5 examples per model that best illustrate a recurring failure mode. For each, write 1-2
sentences on *why* the model likely got it wrong (e.g. bag-of-words baseline can't see negation
scope; BiLSTM's fixed 64-token window truncates a late sentiment-bearing clause; BERT still
misses domain-specific irony common in movie reviews).

In [6]:
# representative_cases = pd.DataFrame([
#     {"model": "baseline", "sentence": "...", "true": 1, "pred": 0, "why": "..."},
#     {"model": "birnn",    "sentence": "...", "true": 0, "pred": 1, "why": "..."},
#     {"model": "bert",     "sentence": "...", "true": 3, "pred": 2, "why": "..."},
# ])
# representative_cases.to_csv(f"results/representative_errors_{DATASET}.csv", index=False)
# representative_cases

## 6. Summary for docs/analysis.md

Once the above is filled in, write 3-5 bullet points here summarizing:
- Which error category is most common overall, and for which model
- Which errors persist across all three models (likely genuinely hard/ambiguous examples,
  possibly annotation noise) vs. which are fixed by more sophisticated architectures
- One concrete proposed fix per major error category (e.g. explicit negation-scope features
  for the baseline, longer max sequence length for BiLSTM, targeted data augmentation for
  sarcasm/irony examples)